<span style="font-weight:bold; font-size: 3rem; color:#333;">- Part 01: Feature Backfill for Train Delay Data (Two-Stage Model)</span>

## 🗒️ Overview

This notebook backfills historical features for the two-stage train delay model.

It performs the following steps:

1. **Choose train stations and time range**: define a list of LocationSignature codes for the Pendeltåg network and select a backfill window (e.g., last X days/months).
2. **Fetch historical TrainAnnouncement data** from Trafikverket's Open API using XML queries.
3. **Fetch auxiliary data** such as weather observations and ReasonCodes (if available) to enrich the dataset.
4. **Engineer features** such as delay minutes, time-of-day, day-of-week, recent delays, weather metrics, and reason categories.
5. **Save the resulting DataFrame** into a Hopsworks feature group for downstream training and inference.

> 🛠️ **Note**: You need to supply a valid Trafikverket API key. The API returns JSON when the request is sent in XML format. Replace placeholders where indicated.


In [9]:
#!pip install -r requirements.txt
#!pip install python-dotenv

In [10]:
import os
import hopsworks
from dotenv import load_dotenv
load_dotenv()

# 1) Where your Hopsworks UI lives (domain in your browser)
os.environ["HOPSWORKS_HOST"] = "eu-west.cloud.hopsworks.ai"
os.environ["HOPSWORKS_PORT"] = "443"

# 2) Hopsworks login key (must be a valid Hopsworks API key)
#print((os.getenv("HOPSWORKS_API_KEY") or "").strip(), os.environ["HOPSWORKS_API_KEY"])
#os.environ["HOPSWORKS_API_KEY"] = (os.getenv("HOPSWORKS_API_KEY") or "").strip()
assert len(os.environ["HOPSWORKS_API_KEY"]) > 20, "Missing/invalid HOPSWORKS_API_KEY in Colab Secrets"

# 3) Your project name (must match exactly in Hopsworks)
#os.environ["HOPSWORKS_PROJECT"] = "Train"

# 4) Trafikverket key (your notebooks use API_KEY_TRAFIK)
#os.environ["API_KEY_TRAFIK"] = (os.getenv("API_TRAIN_TRAFIK") or "").strip()
assert len(os.environ["API_KEY_TRAFIK"]) > 10, "Missing API_TRAIN_TRAFIK in Colab Secrets"






In [11]:
import os
import hopsworks

HOPSWORKS_API_KEY = (os.getenv("HOPSWORKS_API_KEY") or "").strip()
assert HOPSWORKS_API_KEY, "Missing HOPSWORKS_API_KEY secret"
project_name = os.getenv("HOPSWORKS_PROJECT")

HOST = os.environ["HOPSWORKS_HOST"] # must match your browser domain
PORT = 443

conn = hopsworks.connection(
    host=HOST,
    port=PORT,
    api_key_value=HOPSWORKS_API_KEY,
    hostname_verification=True,
)

projects = conn.get_projects()
names = [p.name for p in projects]
print("Projects I can access:", names)



2026-01-02 15:30:33,627 INFO: Closing external client and cleaning up certificates.
2026-01-02 15:30:33,634 INFO: Initializing external client
2026-01-02 15:30:33,636 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


Projects I can access: ['Train', 'Testing']


### 📝 Imports

In [12]:

import datetime
import pandas as pd
import requests
import hopsworks
from typing import List, Dict, Any
 # Added for Colab secrets

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [13]:

HOPSWORKS_API_KEY = os.getenv("HOPSWORKS_API_KEY")
print("HOPSWORKS_API_KEY exists:", HOPSWORKS_API_KEY is not None)
print("HOPSWORKS_API_KEY length:", len(HOPSWORKS_API_KEY.strip()) if HOPSWORKS_API_KEY else None)


HOPSWORKS_API_KEY exists: True
HOPSWORKS_API_KEY length: 81


In [14]:
import hopsworks

conn = hopsworks.connection(
    host=HOST,
    port=PORT,
    api_key_value=HOPSWORKS_API_KEY,
    hostname_verification=True
)

projects = conn.get_projects()
print([p.name for p in projects])


2026-01-02 15:30:35,006 INFO: Closing external client and cleaning up certificates.
2026-01-02 15:30:35,010 INFO: Initializing external client
2026-01-02 15:30:35,012 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


['Train', 'Testing']


## 📡 Connect to Hopsworks Feature Store

In [15]:
project = hopsworks.login(project=project_name, api_key_value=HOPSWORKS_API_KEY)
# Replace 'your_feature_store_name' with the actual name of your feature store
# For this notebook, it's likely something related to 'train_delay_features'
# If your feature store is named after the project name and you just need to ensure it exists,
# you can remove the argument to get_feature_store() or provide the correct project name during login.
# For now, let's assume a feature store named 'train_delay_featurestore' for this notebook's context.
fs = project.get_feature_store()

2026-01-02 15:30:36,345 INFO: Closing external client and cleaning up certificates.
Connection closed.
2026-01-02 15:30:36,350 INFO: Initializing external client
2026-01-02 15:30:36,358 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-01-02 15:30:38,158 INFO: Python Engine initialized.


AttributeError: 'Response' object has no attribute 'error_code'

## 🔑 Configure Trafikverket API and Helper Functions

In [ ]:
# API key for Trafikverket. Replace with your own key or load from an environment variable.
API_KEY_TRAFIK = os.getenv('API_TRAIN_TRAFIK') # Changed to use Colab secrets
TRAFIKVERKET_BASE_URL = 'https://api.trafikinfo.trafikverket.se/v2/data.json'



# Helper to build XML requests
def build_request_xml(api_key: str, object_type: str = 'TrainAnnouncement', limit: int = 10000, extra_filter_xml: str = '', include_fields: list | None = None) -> str:
    include_xml = ''
    if include_fields:
        include_xml = ''.join(f'<INCLUDE>{field}</INCLUDE>' for field in include_fields)
    xml = (
        '<REQUEST>\n'
        f'  <LOGIN authenticationkey="{api_key}" />\n'
        f'  <QUERY objecttype="{object_type}" limit="{limit}" schemaversion="1.0">\n'
        f'{extra_filter_xml}\n'
        f'{include_xml}\n'
        '  </QUERY>\n'
        '</REQUEST>'
    )
    return xml

# Helper to query Trafikverket API
def query_trafikverket(xml_body: str, base_url: str = TRAFIKVERKET_BASE_URL, timeout: int = 30) -> Dict[str, Any]:
    headers = {'Content-Type': 'text/xml'}
    resp = requests.post(base_url, data=xml_body.encode('utf-8'), headers=headers, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

# Build XML filter for a time range
def build_time_filter_xml(start_time: datetime.datetime, end_time: datetime.datetime) -> str:
    start_str = start_time.strftime('%Y-%m-%dT%H:%M:%S')
    end_str = end_time.strftime('%Y-%m-%dT%H:%M:%S')
    return (
        '    <FILTER>\n'
        '      <AND>\n'
        f'        <GT name="AdvertisedTimeAtLocation" value="{start_str}" />\n'
        f'        <LT name="AdvertisedTimeAtLocation" value="{end_str}" />\n'
        '      </AND>\n'
        '    </FILTER>'
    )

# Build station filter
def build_station_filter_xml(station_codes: list) -> str:
    ors = '\n'.join(f'          <EQ name="LocationSignature" value="{code}" />' for code in station_codes)
    return (
        '    <FILTER>\n'
        '      <OR>\n'
        f'{ors}\n'
        '      </OR>\n'
        '    </FILTER>'
    )

# Fetch TrainAnnouncement records
def fetch_announcements(station_codes: list, start_time: datetime.datetime, end_time: datetime.datetime, api_key: str = API_KEY_TRAFIK) -> list:
    time_filter = build_time_filter_xml(start_time, end_time)
    station_filter = build_station_filter_xml(station_codes)
    combined_filter = (
        '    <FILTER>\n'
        '      <AND>\n'
        f'{time_filter}\n'
        f'{station_filter}\n'
        '      </AND>\n'
        '    </FILTER>'
    )
    include_fields = ['AdvertisedTrainIdent', 'AdvertisedTimeAtLocation', 'EstimatedTimeAtLocation', 'LocationSignature', 'Canceled']
    xml = build_request_xml(api_key=api_key, object_type='TrainAnnouncement', limit=100000, extra_filter_xml=combined_filter, include_fields=include_fields)
    data = query_trafikverket(xml)
    try:
        return data['RESPONSE']['RESULT'][0]['TrainAnnouncement']
    except (KeyError, IndexError):
        print('No TrainAnnouncement data returned')
        return []

# Convert records to DataFrame
def records_to_dataframe(records: list) -> pd.DataFrame:
    if not records:
        return pd.DataFrame()
    df = pd.json_normalize(records)
    for col in ['AdvertisedTimeAtLocation', 'EstimatedTimeAtLocation']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
    df['delay_min'] = (df['EstimatedTimeAtLocation'] - df['AdvertisedTimeAtLocation']).dt.total_seconds().div(60)
    df['hour_of_day'] = df['AdvertisedTimeAtLocation'].dt.hour
    df['day_of_week'] = df['AdvertisedTimeAtLocation'].dt.dayofweek
    df = df.rename(columns={'AdvertisedTimeAtLocation': 'event_time'})
    return df

# Placeholder: fetch weather data
def fetch_weather_features(timestamp: datetime.datetime, station_code: str) -> dict:
    return {'temperature': None, 'precipitation': None, 'snow_depth': None}

# Placeholder: fetch reason code data
def fetch_reason_code_features(train_ident: str, timestamp: datetime.datetime) -> dict:
    return {'reason_code': None}

## 🕒 Define Backfill Parameters

In [ ]:
station_codes = ['Cst', 'Sci', 'Mr', 'U', 'Sod', 'Tål']
end_time = datetime.datetime.utcnow()
start_time = end_time - datetime.timedelta(days=30)

print(f'Backfilling from {start_time} to {end_time} for {len(station_codes)} stations')


Backfilling from 2025-12-01 15:33:59.605715 to 2025-12-31 15:33:59.605715 for 6 stations


In [ ]:
# Temporarily redefine fetch_announcements for debugging purposes within this cell.
# This helps diagnose the '400 Bad Request' by printing the XML request and API response text.
# Remember to re-run cell 2393ed3e if you want to revert to the original definition.
def fetch_announcements(station_codes: list, start_time: datetime.datetime, end_time: datetime.datetime, api_key: str = API_KEY_TRAFIK) -> list:
    start_str = start_time.strftime('%Y-%m-%dT%H:%M:%S')
    end_str = end_time.strftime('%Y-%m-%dT%H:%M:%S')

    # Construct the inner parts of the filters without the outer <FILTER> tags
    time_filter_content = (
        f'        <GT name="AdvertisedTimeAtLocation" value="{start_str}" />\n'
        f'        <LT name="AdvertisedTimeAtLocation" value="{end_str}" />'
    )

    station_filter_content = '\n'.join(f'          <EQ name="LocationSignature" value="{code}" />' for code in station_codes)

    # Combine them into a single FILTER block with an AND operator at the top level
    final_filter_xml = (
        '    <FILTER>\n'
        '      <AND>\n'
        f'{time_filter_content}\n'
        '        <OR>\n'
        f'{station_filter_content}\n'
        '        </OR>\n'
        '      </AND>\n'
        '    </FILTER>'
    )

    include_fields = ['AdvertisedTrainIdent', 'AdvertisedTimeAtLocation', 'EstimatedTimeAtLocation', 'LocationSignature', 'Canceled']

    # Use the correctly formatted final_filter_xml
    xml = build_request_xml(api_key=api_key, object_type='TrainAnnouncement', limit=1000, extra_filter_xml=final_filter_xml, include_fields=include_fields)
    print("--- Generated XML Request (for debugging) ---")
    print(xml)
    print("---------------------------------------------")
    try:
        data = query_trafikverket(xml)
        if 'RESPONSE' in data and 'RESULT' in data['RESPONSE'] and len(data['RESPONSE']['RESULT']) > 0 and 'TrainAnnouncement' in data['RESPONSE']['RESULT'][0]:
            return data['RESPONSE']['RESULT'][0]['TrainAnnouncement']
        else:
            print('No TrainAnnouncement data returned or unexpected response structure.')
            print(f"Full API response: {data}") # Print full response to debug
            return []
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error occurred: {e}")
        # Attempt to print the response body for more specific error details from Trafikverket
        print(f"API Response Content: {e.response.text}")
        raise # Re-raise the exception after printing details
    except Exception as e:
        print(f"An unexpected error occurred during API call: {e}")
        raise # Re-raise other exceptions


records = fetch_announcements(station_codes, start_time, end_time)
print(f'Fetched {len(records)} TrainAnnouncement records')

df = records_to_dataframe(records)

weather_rows = []
reason_rows = []
for _, row in df.iterrows():
    weather_rows.append(fetch_weather_features(row['event_time'], row['LocationSignature']))
    reason_rows.append(fetch_reason_code_features(row['AdvertisedTrainIdent'], row['event_time']))
weather_df = pd.DataFrame(weather_rows)
reason_df  = pd.DataFrame(reason_rows)
df = pd.concat([df.reset_index(drop=True), weather_df, reason_df], axis=1)

print('Example row with engineered features:')
df.head()

--- Generated XML Request (for debugging) ---
<REQUEST>
  <LOGIN authenticationkey="9f8266b1879a4f67af11adab988fb151" />
  <QUERY objecttype="TrainAnnouncement" limit="1000" schemaversion="1.0">
    <FILTER>
      <AND>
        <GT name="AdvertisedTimeAtLocation" value="2025-12-01T15:33:59" />
        <LT name="AdvertisedTimeAtLocation" value="2025-12-31T15:33:59" />
        <OR>
          <EQ name="LocationSignature" value="Cst" />
          <EQ name="LocationSignature" value="Sci" />
          <EQ name="LocationSignature" value="Mr" />
          <EQ name="LocationSignature" value="U" />
          <EQ name="LocationSignature" value="Sod" />
          <EQ name="LocationSignature" value="Tål" />
        </OR>
      </AND>
    </FILTER>
<INCLUDE>AdvertisedTrainIdent</INCLUDE><INCLUDE>AdvertisedTimeAtLocation</INCLUDE><INCLUDE>EstimatedTimeAtLocation</INCLUDE><INCLUDE>LocationSignature</INCLUDE><INCLUDE>Canceled</INCLUDE>
  </QUERY>
</REQUEST>
---------------------------------------------

,event_time,AdvertisedTrainIdent,Canceled,LocationSignature,EstimatedTimeAtLocation,delay_min,hour_of_day,day_of_week,temperature,precipitation,snow_depth,reason_code
0,2025-12-31 15:33:00+01:00,639,False,Cst,NaT,NaN,15,2,None,None,None,None
1,2025-12-31 15:31:00+01:00,278,False,Cst,NaT,NaN,15,2,None,None,None,None
2,2025-12-31 15:25:00+01:00,7843,False,Cst,NaT,NaN,15,2,None,None,None,None
3,2025-12-31 15:22:00+01:00,845,False,Cst,2025-12-31 15:23:00+01:00,1.0,15,2,None,None,None,None
4,2025-12-31 15:20:00+01:00,539,False,Cst,NaT,NaN,15,2,None,None,None,None


## 🧬 Create Feature Group and Insert Historical Data

In [ ]:
if df.empty:
    print('No data to insert into feature store.')
else:
    delay_fg = fs.get_or_create_feature_group(
        name='train_delay_features',
        version=1,
        description='Historical train delay features with weather and reason codes',
        primary_key=['LocationSignature', 'event_time'],
        event_time='event_time'
    )
    delay_fg.insert(df, write_options={'wait_for_job': True})
    print('Inserted historical data into feature group "train_delay_features"')


NameError: name 'df' is not defined